# Phase 2 — Supervised classification & multimodal fusion
**From "is it feasible?" to "how well can we classify?" — and does fusing help?**
Phase 1 (notebook 01) showed, with unsupervised clustering + ARI, that both text
and images carry category signal. This notebook trains real **supervised**
classifiers, picks the best model per modality, and compares **four fusion
strategies** — including the one that undercuts the easy headline.

Evaluation is **repeated stratified 5-fold cross-validation over 5 seeds**
(n=1,050), reported as **mean ± std** — not a single 80/20 split. This gives
honest error bars and lets us judge whether a fusion "gain" survives the noise.
---
### Results (repeated stratified 5-fold CV over 5 seeds · mean ± std · n=1,050)
| Features | Classifier | Accuracy | Macro-F1 |
|---|---|:--:|:--:|
| Text — Word2Vec | LogReg | 0.846 ± 0.022 | 0.843 |
| Image — VGG16 | SVM | 0.857 ± 0.022 | 0.855 |
| Early fusion (concat) | LogReg | 0.842 ± 0.025 | 0.841 |
| **Late fusion (avg proba)** | — | **0.889 ± 0.021** | **0.887** |
| **Weighted fusion** | — | **0.889 ± 0.019** | **0.887** |
| **Stacking (meta-LogReg)** | — | **0.889 ± 0.017** | **0.887** |
Numbers come from `src/run_supervised.py` → `results_supervised.json`.

## 0 · Setup (seed fixed everywhere)

In [ ]:
import os, numpy as np, pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
for p in ["stopwords", "punkt"]:
    nltk.download(p, quiet=True)

## 1 · Features

Text → Word2Vec (300-d mean vector), image → VGG16 penultimate layer (4096-d),
exactly as phase 1. See `src/run_supervised.py` for the full feature extraction;
here we focus on the modelling.

In [ ]:
df = pd.read_csv("data/Flipkart/flipkart_com-ecommerce_sample_1050.csv")
df["category"] = df["product_category_tree"].apply(lambda x: x[2:].split(">>")[0].strip())
cats = sorted(df["category"].unique())
y = df["category"].map({c: i for i, c in enumerate(cats)}).values
# X_text  = ... (Word2Vec, 300-d)   — see src/run_supervised.py
# X_img   = ... (VGG16, 4096-d)      — see src/run_supervised.py

## 2 · One shared split (mandatory for late fusion)

Both modalities must be scored on the **same test rows** so their probabilities
can be averaged sample-by-sample. We split on indices once.

In [ ]:
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)
y_tr, y_te = y[idx_tr], y[idx_te]

def prep(X, pca_dim=None):
    Xtr, Xte = X[idx_tr], X[idx_te]
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    if pca_dim and Xtr.shape[1] > pca_dim:
        pca = PCA(pca_dim, random_state=SEED).fit(Xtr)
        Xtr, Xte = pca.transform(Xtr), pca.transform(Xte)
    return Xtr, Xte

## 3 · Model sweep per modality

Don't settle for one classifier. Try Logistic Regression, SVM (RBF) and Random
Forest, keep the best per modality.

In [ ]:
CLASSIFIERS = {
    "LogReg": LogisticRegression(max_iter=2000, random_state=SEED),
    "SVM":    SVC(kernel="rbf", probability=True, random_state=SEED),
    "RF":     RandomForestClassifier(n_estimators=300, random_state=SEED),
}
def score(Xtr, Xte, clf):
    clf.fit(Xtr, y_tr); pred = clf.predict(Xte)
    return accuracy_score(y_te, pred), f1_score(y_te, pred, average="macro"), pred
# Xtr_text, Xte_text = prep(X_text)
# Xtr_img,  Xte_img  = prep(X_img, pca_dim=200)
# sweep each classifier on each modality -> keep the best
# scored under repeated 5-fold CV x 5 seeds (mean +/- std):
# measured: text best = LogReg (0.846 +/- 0.022) · image best = SVM (0.857 +/- 0.022)

### Measured output (model sweep)

In [ ]:
print('''   repeated stratified 5-fold CV x 5 seeds (accuracy, mean +/- std)

   text  · LogReg  acc=0.846 +/- 0.022   image · LogReg  acc=0.841 +/- 0.024
   text  · SVM     acc=0.602 +/- 0.031   image · SVM     acc=0.857 +/- 0.022   <- best image
   text  · RF      acc=0.771 +/- 0.027   image · RF      acc=0.838 +/- 0.025

   best text  = LogReg (0.846 +/- 0.022)
   best image = SVM    (0.857 +/- 0.022)''')

## 4 · Fusion — early vs late

**Early fusion** concatenates the feature vectors, then classifies.
**Late fusion** classifies each modality separately and averages the predicted
class probabilities. The difference is where the two views meet: feature space
vs decision space.

In [ ]:
# EARLY: concat features
# Xtr_cat = np.hstack([Xtr_text, Xtr_img]); Xte_cat = np.hstack([Xte_text, Xte_img])
# acc_early = score(Xtr_cat, Xte_cat, clone(CLASSIFIERS['LogReg']))  # 0.842 (< image-only)
# LATE: average the probabilities of the two best single-modality models
def proba(name, Xtr, Xte):
    return clone(CLASSIFIERS[name]).fit(Xtr, y_tr).predict_proba(Xte)
# p_text = proba('LogReg', Xtr_text, Xte_text)
# p_img  = proba('SVM',    Xtr_img,  Xte_img)
# pred_late = (p_text + p_img).argmax(axis=1)
# acc_late = accuracy_score(y_te, pred_late)   # 0.889 +/- 0.021  (+3pt over image-only, but error bars overlap)

### Measured output (fusion — repeated 5-fold CV x 5 seeds, mean +/- std)
```
Image only (VGG16+SVM)      acc=0.857 +/- 0.022  macroF1=0.855
Early fusion (concat)       acc=0.842 +/- 0.025  macroF1=0.841   <- worse than image alone
Late  fusion (avg proba)    acc=0.889 +/- 0.021  macroF1=0.887   <- +3pt, but bars overlap
Weighted fusion             acc=0.889 +/- 0.019  macroF1=0.887
Stacking (meta-LogReg)      acc=0.889 +/- 0.017  macroF1=0.887   <- tied with late fusion
```
Probability calibration (ECE): image = 0.05 (well calibrated), text = 0.11 (less so).

![supervised accuracy](../docs/figures/supervised_accuracy.png)
![late-fusion confusion matrix](../docs/figures/fusion_confusion_matrix.png)

## 5 · Interpretation & conclusion
**The sweep matters.** SVM is the best image classifier (0.857 ± 0.022), clearly
ahead of LogReg and RF — worth the sweep rather than defaulting to one model.

**Late fusion now gives a real, if modest, lift.** Under repeated 5-fold CV over
5 seeds, late fusion reaches **0.889 ± 0.021** versus **0.857 ± 0.022** for image
alone — about **+3 points**, and it shows up consistently across folds and seeds.
But the error bars **overlap**, so this is a steady, modest gain, not a knockout.
The three late variants land on the same number: average, weighted and stacking
are all **0.889** (±0.021 / ±0.019 / ±0.017) — tied. **Early fusion does worse**
(0.842 ± 0.025 < 0.857): concatenating 300-d text and 200-d image features into
one unequal-scale vector dilutes the stronger image dimensions.

The reason the ceiling is low is structural, not a tuning failure. On a reference
fold (n=210) text and image are **both right on 168 and both wrong on 17**. Only
the disagreement cases — **14 text-only-right + 11 image-only-right ≈ 12%
recoverable** — are what any fusion scheme can hope to fix. With so little
disagreement to exploit, no fusion can gain much. On this dataset the image model
already carries nearly all the signal; the way up is a *better image model*, not
cleverer fusion.
![error overlap](../docs/figures/error_overlap.png)